In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from show import *
from utils import *
from datetime import datetime

In [2]:
files = sorted(glob.glob('/home/ulyanov/data/solo/phi/polar/*'))
#files = sorted(glob.glob('/home/ulyanov/data/sdo/hmi/polar/*'))
print(len(files))
print(files[0], files[-1])

283
/home/ulyanov/data/solo/phi/polar/solo_L2_phi-fdt-blos_20250426T000003_V202602220258_0544260501.fits.gz /home/ulyanov/data/solo/phi/polar/solo_L2_phi-fdt-blos_20250531T182003_V202602220334_0545310507.fits.gz


In [3]:
nx, ny = 512, 512
xc, yc = nx // 2 - 0.5, ny // 2 - 0.5
Rsun = nx // 2

view_north = View(nx, ny, xc, yc, Rsun, -90, 90, 0) ### North pole view
grid = np.mgrid[:nx,:ny].astype(np.float32)
xi, yi = (grid[0] - xc) / Rsun, (grid[1] - yc) / Rsun

binning = 1
r2 = rebin(xi ** 2 + yi ** 2, binning)

data_ = None
alpha_ = None
time_ = None
coverage = 0
a, b = 0, 0


for file in files[:]:
    print(file)

    with fits.open(file) as hdul:
        header = hdul[0].header.copy()
        data = hdul[0].data.copy()

    #data = rebin(data, 4, update_header=header)
    view = View.from_header(header)

    transform = view_north.to_spherical(correct_mu=True, mu_thr=0.1) - view.to_spherical(correct_mu=True, correct_dr=True, mu_thr=0.1)
    grid_, alpha = transform(grid)
    data = bilinear(data, *grid_) * alpha
    time = datetime.fromisoformat(header['DATE-OBS'])

    if data_ is not None:
        timedelta = (time - time_).total_seconds() / 24 / 3600
        delta_x, delta_y = np.gradient((data + data_) / 2)

        delta = rebin(delta_x * yi - delta_y * xi, binning)
        delta_t = rebin(data - data_, binning) / timedelta

        n = (~np.isnan(delta_t)) / rebin(alpha * alpha_, binning)
        coverage += np.nan_to_num(n)

        a += np.nan_to_num((delta ** 2 - a) * n / coverage)
        b += np.nan_to_num((delta_t * delta - b) * n / coverage)

    data_ = data.copy()
    alpha_ = alpha.copy()
    time_ = time


#mean[coverage == 0] = np.nan
#coverage[coverage == 0] = np.nan

/home/ulyanov/data/solo/phi/polar/solo_L2_phi-fdt-blos_20250426T000003_V202602220258_0544260501.fits.gz
/home/ulyanov/data/solo/phi/polar/solo_L2_phi-fdt-blos_20250426T020003_V202602220258_0544260502.fits.gz
/home/ulyanov/data/solo/phi/polar/solo_L2_phi-fdt-blos_20250426T040004_V202602220258_0544260503.fits.gz
/home/ulyanov/data/solo/phi/polar/solo_L2_phi-fdt-blos_20250426T060004_V202602220258_0544260504.fits.gz
/home/ulyanov/data/solo/phi/polar/solo_L2_phi-fdt-blos_20250426T080004_V202602220258_0544260505.fits.gz
/home/ulyanov/data/solo/phi/polar/solo_L2_phi-fdt-blos_20250426T100003_V202602220258_0544260506.fits.gz
/home/ulyanov/data/solo/phi/polar/solo_L2_phi-fdt-blos_20250426T120004_V202602220258_0544260507.fits.gz
/home/ulyanov/data/solo/phi/polar/solo_L2_phi-fdt-blos_20250426T140004_V202602220258_0544260508.fits.gz
/home/ulyanov/data/solo/phi/polar/solo_L2_phi-fdt-blos_20250426T160004_V202602220258_0544260509.fits.gz
/home/ulyanov/data/solo/phi/polar/solo_L2_phi-fdt-blos_20250426T

In [13]:
with fits.open(file) as hdul:
    header = hdul[0].header.copy()
    data = hdul[0].data.copy()

data1 = remap(data, header, dlat=0.2, dlon=0.2, correct_mu=True, correct_dr=False, mu_thr=0.1)

In [14]:
plt.figure(figsize=(10,10))
plt.imshow(data1)

In [4]:
W = b / a / Rsun * 180 / np.pi

plt.figure(figsize=(10,10))
plt.imshow(W, vmin=-5, vmax=5)
plt.tight_layout()

/tmp/ipykernel_23631/1092491534.py:1: RuntimeWarning: invalid value encountered in divide
  W = b / a / Rsun * 180 / np.pi


In [5]:
dr = 0.01
ri = np.arange(0, 1, dr)
Q = []

for r in ri:
    t = np.where(np.abs(r2 - (r + dr / 2)) < dr / 2)
    q = np.mean(b[t]) / np.mean(a[t])
    #q = np.mean(b[t] / a[t])
    Q += [q]

Q = np.array(Q) / Rsun * 180 / np.pi

plt.figure(figsize=(10,8))
plt.plot(1 - (ri + dr / 2), Q)

In [9]:
p = np.polyfit(1 - (ri + dr / 2), Q, deg=2)
print(p)

plt.figure(figsize=(10,8))
plt.scatter(1 - r2, W, s=a / 50)
#plt.plot(np.arange(0, 1, 0.01), np.polyval([1.787, 2.396, -0.529], np.arange(0, 1, 0.01)), color='tab:red', ls='--')
plt.plot(1 - (ri + dr / 2), Q, color='black', ls='-.', lw=1)
plt.plot(np.arange(0, 1.01, 0.01), np.polyval(p, np.arange(0, 1.01, 0.01)), color='black')

plt.xlim(0, 1)
plt.ylim(-3, 3)
plt.tight_layout()

[ 2.94313788 -2.06625126 -0.3371357 ]


In [9]:
np.sin(60 * np.pi / 180) ** 2

np.float64(0.7499999999999999)

In [6]:
1 / (1 / 25.38 - 1 / 27.2753)

365.2440848414497

In [7]:
1 / (1 / 27.2753 + 1 / 365.256363)

25.380059283864444